
# 08 - Model Benchmarking ⚡

## Objetivo
Comparar diferentes modelos para predicción de riesgo de outage.

## Modelos:
- Logistic Regression
- ANN (MLP)
- XGBoost

## Métrica principal:
- Precision Top-K


In [1]:

import pandas as pd

df = pd.read_parquet("../data/processed/dataset_with_events.parquet")
df = df.sort_values(["Meter ID","Datetime"])


In [4]:

# Target (t+3)
df["target"] = (
    df.groupby("Meter ID")["has_outage"]
    .rolling(3)
    .max()
    .shift(-3)
    .reset_index(level=0, drop=True)
)

df["target"] = df["target"].fillna(0).astype(int)


In [6]:
# =========================
# FEATURES AVANZADAS
# =========================

df = df.sort_values(["Meter ID","Datetime"])

# tendencias
df["Voltage_trend"] = df.groupby("Meter ID")["Voltage_avg"].diff()
df["Current_trend"] = df.groupby("Meter ID")["Current_avg"].diff()

# rolling (1h = 4 intervalos)
df["Voltage_std_1h"] = (
    df.groupby("Meter ID")["Voltage_avg"]
    .rolling(4).std()
    .reset_index(level=0, drop=True)
)

df["Current_std_1h"] = (
    df.groupby("Meter ID")["Current_avg"]
    .rolling(4).std()
    .reset_index(level=0, drop=True)
)

# min/max
df["Voltage_min_1h"] = (
    df.groupby("Meter ID")["Voltage_avg"]
    .rolling(4).min()
    .reset_index(level=0, drop=True)
)

df["Voltage_max_1h"] = (
    df.groupby("Meter ID")["Voltage_avg"]
    .rolling(4).max()
    .reset_index(level=0, drop=True)
)

In [7]:

# Features (usar las avanzadas)
features = [
    "Voltage_avg","Voltage_unbalance","Voltage_min","Voltage_max",
    "Current_avg","Current_unbalance","Power_kW","PF",
    "Sag_pct","Swell_pct","Disturbance_score",
    "Voltage_drop","Current_spike",
    "Voltage_trend","Current_trend",
    "Voltage_std_1h","Current_std_1h",
    "Voltage_min_1h","Voltage_max_1h"
]

df_model = df.dropna(subset=features)

X = df_model[features]
y = df_model["target"]


In [8]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


## 🔥 Función evaluación Top-K

In [9]:

def evaluate_top_k(model, X_test, y_test, k=1000):
    probs = model.predict_proba(X_test)[:,1]

    df_eval = X_test.copy()
    df_eval["target"] = y_test.values
    df_eval["risk_score"] = probs

    top = df_eval.sort_values("risk_score", ascending=False).head(k)

    precision = top["target"].mean()
    outages = top["target"].sum()

    return precision, outages


## ⚡ 1. Logistic Regression

In [10]:

from sklearn.linear_model import LogisticRegression

model_lr = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

model_lr.fit(X_train, y_train)

lr_precision, lr_outages = evaluate_top_k(model_lr, X_test, y_test)

print("Logistic Regression:")
print("Precision:", lr_precision)
print("Outages:", lr_outages)


c:\Users\rroman\OneDrive - Empresa Eléctrica de Guatemala, S.A. (EEGSA)\Documents\Unidad\2026\Capacitación\Data Science y Machine Learning\proyecto final\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression:
Precision: 0.006
Outages: 6


## ⚡ 2. ANN (MLP)

In [11]:

from sklearn.neural_network import MLPClassifier

model_ann = MLPClassifier(
    hidden_layer_sizes=(64,32),
    max_iter=50
)

model_ann.fit(X_train, y_train)

ann_precision, ann_outages = evaluate_top_k(model_ann, X_test, y_test)

print("ANN:")
print("Precision:", ann_precision)
print("Outages:", ann_outages)


ANN:
Precision: 0.001
Outages: 1


## ⚡ 3. XGBoost

In [12]:

from xgboost import XGBClassifier

model_xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model_xgb.fit(X_train, y_train)

xgb_precision, xgb_outages = evaluate_top_k(model_xgb, X_test, y_test)

print("XGBoost:")
print("Precision:", xgb_precision)
print("Outages:", xgb_outages)


XGBoost:
Precision: 0.076
Outages: 76


## 🏆 Comparación Final

In [13]:

results = pd.DataFrame({
    "Model": ["Logistic", "ANN", "XGBoost"],
    "Precision_TopK": [lr_precision, ann_precision, xgb_precision],
    "Outages_detected": [lr_outages, ann_outages, xgb_outages]
})

results.sort_values("Precision_TopK", ascending=False)


,Model,Precision_TopK,Outages_detected
2,XGBoost,0.076,76
0,Logistic,0.006,6
1,ANN,0.001,1
